In [1]:
import pandas as pd
from data_preprocessing import create_train_test_val_sets, get_processed_df
import joblib
import os
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.feature_selection import SelectKBest, VarianceThreshold, chi2, f_classif, mutual_info_classif
from sklearn.base import clone
from collections import Counter
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"..\data\raw\Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"..\data\raw\dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

----------Processing None Dataset----------

Class Distribution:
Label
0    0.518415
1    0.481585
Name: proportion, dtype: float64
int64

Total Missing Values: 0
No categorical features to hash
Shape After Processing: (247950, 42)
True
----------Processing None Dataset----------

Class Distribution:
Label
legitimate    0.5
phishing      0.5
Name: proportion, dtype: float64
object

Total Missing Values: 0
Shape After Processing: (11430, 32856)
True
Train/validation/test split prepared: 179143 instances for training, 31614 instances for validation, 37193 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 8257 instances for training, 1458 instances for validation, 1715 instances for testing
Stratified 5-fold CV splits created.


In [3]:
kaggle_sets["y_train"] = kaggle_sets["y_train"].astype(int)
kaggle_sets["y_val"] = kaggle_sets["y_val"].astype(int)
kaggle_sets["y_test"] = kaggle_sets["y_test"].astype(int)

In [4]:
#import phase 1 models
xgb_mendeley = joblib.load('./models/phase_1/xgboost_mendeley_no_fs.joblib')
xgb_kaggle = joblib.load('./models/phase_1/xgboost_kaggle_no_fs.joblib')
logreg_mendeley = joblib.load('./models/phase_1/logreg_mendeley_no_fs.joblib')
logreg_kaggle = joblib.load('./models/phase_1/logreg_kaggle_no_fs.joblib')
rf_mendeley = joblib.load('./models/phase_1/rf_mendeley_no_fs.joblib')
rf_kaggle = joblib.load('./models/phase_1/rf_kaggle_no_fs.joblib')

In [5]:
def run_stability_algorithm(ds_name, ds_set, model,  fs_params,fs_method='anova', stability_threshold=0.8):
    """
    Uses pre-calculated cv_splits to determine feature stability.
    """
    x_train_full = ds_set["x_train"]
    y_train_full = ds_set["y_train"]
    splits = ds_set["cv_splits"]
    
    numeric_cols = [col for col in x_train_full.columns if not col.startswith("hash")]
    hash_cols = [col for col in x_train_full.columns if col.startswith("hash")]

    fold_selected_features = []
    
    # Define FS Strategies
    selectors = {
        'chi2': chi2,
        'anova': f_classif,
        'mi': mutual_info_classif,
        'variance': lambda v: VarianceThreshold(threshold=v)
    }

    print(f"--- Stability Search: {ds_name} | FS: {fs_method} | Model: {model.__class__.__name__} ---")
    
    
    #Iterate through pre-defined splits
    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        # Create fold-specific data using iloc for index-based slicing
        x_fold_train = x_train_full.iloc[train_idx]
        y_fold_train = y_train_full.iloc[train_idx]
        x_train_keep = None
        #Apply the chosen FS method
        if fs_method == 'variance':
            selector = VarianceThreshold(threshold=fs_params)
            selector.fit(x_fold_train) 
        else:
            if fs_method == 'chi2' and ds_name=="Kaggle":
                # Ensure categorical features for chi2
                x_train_keep = x_fold_train[numeric_cols]
                x_fold_train = x_fold_train[hash_cols]
                
            score_func = selectors[fs_method]
            selector = SelectKBest(score_func=score_func, k=fs_params)
            selector.fit(x_fold_train, y_fold_train)

        # Record winning features for this fold
        selected_features = x_fold_train.columns[selector.get_support()].tolist()
        
                   
        if x_train_keep is not None:
            selected_features = list(x_train_keep.columns) + selected_features
        fold_selected_features.append(selected_features)

    #Compute Stability Metrics
    all_fold_features = [f for sublist in fold_selected_features for f in sublist] #flatten list of lists
    counts = Counter(all_fold_features) #dict of feature: count across folds
    num_folds = len(splits)
    print(num_folds)

    #save stability score for each feature
    stability_scores = {feature: count / num_folds for feature, count in counts.items()}
    #Re-build the dictionary by removing anything under the threshold
    stability_scores = {feat: score for feat, score in stability_scores.items() if score >= stability_threshold}

    #Extract just the names for 
    stable_features = list(stability_scores.keys())
    print(f"Stable Features ({len(stable_features)}): {stable_features}")
    
    #Final Training on the Stable subset
    model.fit(x_train_full[stable_features], y_train_full)
    
    # Evaluate on Validation Set
    y_val_pred = model.predict(ds_set["x_val"][stable_features])
    val_f1 = f1_score(ds_set["y_val"], y_val_pred)


    return {
        "method": fs_method,
        "stable_features": stable_features,
        "val_f1": val_f1,
        "stability_scores": stability_scores
    }

In [6]:
def save_stability_report(model_name, ds_name, all_results):
    """
    Saves the stability report to a CSV file.
    """
    final_summary=[]
    for results in all_results:    
        summary_results = {
            'FS_Method': results['method'],
            'Avg_Val_F1': results['val_f1'],
            'Num Stable Features': len(results['stable_features']),
            'Stable features': results['stability_scores']
        }
        final_summary.append(summary_results)
    
    # Ensure the output directory exists
    os.makedirs("../results/phase_3/stability_reports", exist_ok=True)
    summary_df = pd.DataFrame(final_summary)
    filename = f"../results/phase_3/stability_reports/{model_name}_{ds_name}_stability_report.csv"
    summary_df.to_csv(filename, index=False)

In [7]:
#------------MENDELEY DATASET AND XGBOOST MODEL--------------
fs_params_mendeley_xgb = {
    'anova': 30,
    'chi2': 25,
    'mi': 25,
    'variance': 0.05
}
all_results = []
for fs, k in fs_params_mendeley_xgb.items():
    model_name = "XGBoost"
    # Run the algorithm
    res = run_stability_algorithm("Mendeley", mendeley_sets,xgb_mendeley, fs_params=k, fs_method=fs)
    all_results.append(res)

#save the report
save_stability_report(model_name, "Mendeley", all_results)

#----------MENDELEY DATASET AND LOGISTIC REGRESSION MODEL--------------
fs_params_mendeley_logreg = { #UPDATE THESE
    'anova': 20,
    'chi2': 15,
    'mi': 20,
    'variance': 0.05
}
all_results_lr = []
for fs, k in fs_params_mendeley_logreg.items():
    model_name = "Logistic Regression"
    # Run the algorithm
    results = run_stability_algorithm("Mendeley", mendeley_sets,logreg_mendeley, fs_params=k, fs_method=fs)
    all_results_lr.append(results)
#save the report
save_stability_report(model_name, "Mendeley", all_results_lr)

#----------MENDELEY DATASET AND RANDOM FOREST MODEL--------------
fs_params_mendeley_rf = {#UPDATE THESE
    'anova': 25,
    'chi2': 20,
    'mi': 25,
    'variance': 0.05
}
all_results_rf = []
for fs, k in fs_params_mendeley_rf.items():
    model_name = "Random Forest"
    # Run the algorithm
    results = run_stability_algorithm("Mendeley", mendeley_sets,rf_mendeley, fs_params=k, fs_method=fs)     
    all_results_rf.append(results)
#save the report
save_stability_report(model_name, "Mendeley", all_results_rf)

   

--- Stability Search: Mendeley | FS: anova | Model: XGBClassifier ---


c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


5
Stable Features (30): ['url_length', 'number_of_dots_in_url', 'having_repeated_digits_in_url', 'number_of_digits_in_url', 'number_of_special_char_in_url', 'number_of_hyphens_in_url', 'number_of_slash_in_url', 'number_of_questionmark_in_url', 'number_of_equal_in_url', 'number_of_at_in_url', 'number_of_dollar_in_url', 'number_of_percent_in_url', 'domain_length', 'number_of_dots_in_domain', 'number_of_hyphens_in_domain', 'having_special_characters_in_domain', 'number_of_special_characters_in_domain', 'having_digits_in_domain', 'number_of_digits_in_domain', 'having_repeated_digits_in_domain', 'number_of_subdomains', 'average_subdomain_length', 'number_of_special_characters_in_subdomain', 'having_digits_in_subdomain', 'number_of_digits_in_subdomain', 'path_length', 'having_query', 'having_anchor', 'entropy_of_url', 'entropy_of_domain']


KeyboardInterrupt: 

In [ ]:
#------------KAGGLE DATASET AND XGBOOST MODEL--------------
fs_params_kaggle_xgb = {
    'anova': 25000,
    'chi2': 30000,
    'mi': 25000,
    'variance': 0.1
}
all_results = []
for fs, k in fs_params_kaggle_xgb.items():
    model_name = "XGBoost"
    # Run the algorithm
    res = run_stability_algorithm("Kaggle", kaggle_sets, xgb_kaggle, fs_params=k, fs_method=fs)
    all_results.append(res)

#save the report
save_stability_report(model_name, "Kaggle", all_results)

#----------KAGGLE DATASET AND LOGISTIC REGRESSION MODEL--------------
fs_params_kaggle_logreg = { #UPDATE THESE
    'anova': 25000,
    'chi2': 30000,
    'mi': 25000,
    'variance': 0.05
}
all_results_lr = []
for fs, k in fs_params_kaggle_logreg.items():
    model_name = "Logistic Regression"
    # Run the algorithm
    results = run_stability_algorithm("Kaggle", kaggle_sets,logreg_kaggle, fs_params=k, fs_method=fs)
    all_results_lr.append(results)
#save the report
save_stability_report(model_name, "Kaggle", all_results_lr)

#----------KAGGLE DATASET AND RANDOM FOREST MODEL--------------
fs_params_kaggle_rf = {#UPDATE THESE
    'anova': 25000,
    'chi2': 25000,
    'mi': 30000,
    'variance': 0.05
}
all_results_rf = []
for fs, k in fs_params_kaggle_rf.items():
    model_name = "Random Forest"
    # Run the algorithm
    results = run_stability_algorithm("Kaggle", kaggle_sets,rf_kaggle, fs_params=k, fs_method=fs)     
    all_results_rf.append(results)
#save the report
save_stability_report(model_name, "Kaggle", all_results_rf)

   

--- Stability Search: Kaggle | FS: anova | Model: Pipeline ---


c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


5
Stable Features (25): ['ip', 'nb_qm', 'nb_www', 'ratio_digits_url', 'phish_hints', 'nb_hyperlinks', 'domain_in_title', 'domain_age', 'google_index', 'page_rank', 'hash_2901', 'hash_4122', 'hash_5592', 'hash_7445', 'hash_8367', 'hash_11374', 'hash_13601', 'hash_16612', 'hash_16898', 'hash_19852', 'hash_23425', 'hash_29497', 'hash_29991', 'hash_30955', 'hash_31137']
--- Stability Search: Kaggle | FS: chi2 | Model: Pipeline ---
5
Stable Features (115): ['length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_redirection', 'nb_extern

c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


5
Stable Features (238): ['length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_at', 'nb_qm', 'nb_and', 'nb_eq', 'nb_slash', 'nb_www', 'nb_com', 'ratio_digits_url', 'ratio_digits_host', 'tld_in_subdomain', 'prefix_suffix', 'length_words_raw', 'shortest_word_host', 'longest_words_raw', 'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path', 'phish_hints', 'statistical_report', 'nb_hyperlinks', 'ratio_intHyperlinks', 'ratio_extRedirection', 'external_favicon', 'links_in_tags', 'ratio_intMedia', 'safe_anchor', 'empty_title', 'domain_in_title', 'domain_with_copyright', 'domain_registration_length', 'domain_age', 'google_index', 'page_rank', 'hash_468', 'hash_688', 'hash_773', 'hash_808', 'hash_836', 'hash_916', 'hash_1373', 'hash_1477', 'hash_1480', 'hash_2024', 'hash_2029', 'hash_2060', 'hash_2249', 'hash_2306', 'hash_2386', 'hash_2671', 'hash_2901', 'hash_2932', 'hash_3070', 'hash_3079', 'hash_3156', 'hash_3366', 'hash_3438', 'hash_3607', 'hash_3931', 'hash_4122', 'hash_4